In [ ]:
!nvidia-smi

Mon Jul  6 01:29:39 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   39C    P8             13W /   70W |       0MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

In [ ]:
import torch, torch.nn as nn
from easydict import EasyDict
from models.dgcnn_group import DGCNN_Grouper
from models.Transformer import PCTransformer
from models.PoinTr import PoinTr, Fold, fps

# --- grouper: orijinalin BIREBIR kopyasi (input_trans 6D), sarmalama yok ---
def grouper_init(self):
    nn.Module.__init__(self)
    self.input_trans = nn.Conv1d(6, 8, 1)
    self.layer1 = nn.Sequential(nn.Conv2d(16,32,1,bias=False),  nn.GroupNorm(4,32),  nn.LeakyReLU(0.2))
    self.layer2 = nn.Sequential(nn.Conv2d(64,64,1,bias=False),  nn.GroupNorm(4,64),  nn.LeakyReLU(0.2))
    self.layer3 = nn.Sequential(nn.Conv2d(128,64,1,bias=False), nn.GroupNorm(4,64),  nn.LeakyReLU(0.2))
    self.layer4 = nn.Sequential(nn.Conv2d(128,128,1,bias=False),nn.GroupNorm(4,128), nn.LeakyReLU(0.2))
def grouper_forward(self, x):
    coor = x[:, :3].contiguous(); f = self.input_trans(x)
    f = self.get_graph_feature(coor,f,coor,f); f=self.layer1(f); f=f.max(-1)[0]
    cq,fq = self.fps_downsample(coor,f,512); f=self.get_graph_feature(cq,fq,coor,f); f=self.layer2(f); f=f.max(-1)[0]; coor=cq
    f = self.get_graph_feature(coor,f,coor,f); f=self.layer3(f); f=f.max(-1)[0]
    cq,fq = self.fps_downsample(coor,f,128); f=self.get_graph_feature(cq,fq,coor,f); f=self.layer4(f); f=f.max(-1)[0]; coor=cq
    return coor, f
DGCNN_Grouper.__init__ = grouper_init
DGCNN_Grouper.forward  = grouper_forward

# --- PoinTr: orijinalin BIREBIR kopyasi + color_head, sarmalama yok ---
def poinTr_init(self, config, **kw):
    nn.Module.__init__(self)
    self.trans_dim=config.trans_dim; self.knn_layer=config.knn_layer
    self.num_pred=config.num_pred;  self.num_query=config.num_query
    self.fold_step=int(pow(self.num_pred//self.num_query,0.5)+0.5)
    self.base_model=PCTransformer(in_chans=3, embed_dim=self.trans_dim, depth=[6,8], drop_rate=0., num_query=self.num_query, knn_layer=self.knn_layer)
    self.foldingnet=Fold(self.trans_dim, step=self.fold_step, hidden_dim=256)
    self.color_head=Fold(self.trans_dim, step=self.fold_step, hidden_dim=256)
    self.increase_dim=nn.Sequential(nn.Conv1d(self.trans_dim,1024,1), nn.BatchNorm1d(1024), nn.LeakyReLU(0.2), nn.Conv1d(1024,1024,1))
    self.reduce_map=nn.Linear(self.trans_dim+1027, self.trans_dim)
def poinTr_forward(self, xyz):
    q, coarse = self.base_model(xyz); B,M,C = q.shape
    gf = self.increase_dim(q.transpose(1,2)).transpose(1,2); gf = torch.max(gf,1)[0]
    rf = self.reduce_map(torch.cat([gf.unsqueeze(-2).expand(-1,M,-1), q, coarse],-1).reshape(B*M,-1))
    reb_xyz = (self.foldingnet(rf).reshape(B,M,3,-1) + coarse.unsqueeze(-1)).transpose(2,3).reshape(B,-1,3)
    reb_rgb = torch.sigmoid(self.color_head(rf)).reshape(B,M,3,-1).transpose(2,3).reshape(B,-1,3)
    reb = torch.cat([reb_xyz, reb_rgb], -1)
    inp = fps(xyz[:,:,:3].contiguous(), self.num_query)
    return torch.cat([coarse, inp],1), torch.cat([reb, xyz],1)
PoinTr.__init__ = poinTr_init
PoinTr.forward  = poinTr_forward

cfg = EasyDict(trans_dim=384, knn_layer=1, num_pred=6144, num_query=96)
model_c = PoinTr(cfg)
for k in ["base_model.grouper.input_trans.weight","base_model.grouper.input_trans.bias"]:
    base.pop(k, None)
mi, ui = model_c.load_state_dict(base, strict=False)
print(f"OK | missing={len(mi)} unexpected={len(ui)}")
model_c.to(DEV)


2026-07-06 03:37:50,181 - MODEL - INFO -  Transformer with knn_layer 1


OK | missing=30 unexpected=0


PoinTr(
  (base_model): PCTransformer(
    (grouper): DGCNN_Grouper(
      (input_trans): Conv1d(6, 8, kernel_size=(1,), stride=(1,))
      (layer1): Sequential(
        (0): Conv2d(16, 32, kernel_size=(1, 1), stride=(1, 1), bias=False)
        (1): GroupNorm(4, 32, eps=1e-05, affine=True)
        (2): LeakyReLU(negative_slope=0.2)
      )
      (layer2): Sequential(
        (0): Conv2d(64, 64, kernel_size=(1, 1), stride=(1, 1), bias=False)
        (1): GroupNorm(4, 64, eps=1e-05, affine=True)
        (2): LeakyReLU(negative_slope=0.2)
      )
      (layer3): Sequential(
        (0): Conv2d(128, 64, kernel_size=(1, 1), stride=(1, 1), bias=False)
        (1): GroupNorm(4, 64, eps=1e-05, affine=True)
        (2): LeakyReLU(negative_slope=0.2)
      )
      (layer4): Sequential(
        (0): Conv2d(128, 128, kernel_size=(1, 1), stride=(1, 1), bias=False)
        (1): GroupNorm(4, 128, eps=1e-05, affine=True)
        (2): LeakyReLU(negative_slope=0.2)
      )
    )
    (pos_embed): Sequent

In [ ]:
!pip install "numpy==1.26.4" --force-reinstall --no-deps

  Using cached numpy-1.26.4-cp312-cp312-manylinux_2_17_x86_64.manylinux2014_x86_64.whl.metadata (61 kB)
Using cached numpy-1.26.4-cp312-cp312-manylinux_2_17_x86_64.manylinux2014_x86_64.whl (18.0 MB)
  Attempting uninstall: numpy
    Found existing installation: numpy 1.26.4
    Uninstalling numpy-1.26.4:
      Successfully uninstalled numpy-1.26.4


In [ ]:
import os
os.environ["CUDA_HOME"] = "/usr/local/cuda-11.8"
os.environ["PATH"] = "/usr/local/cuda-11.8/bin:" + os.environ["PATH"]
%cd /content/PoinTr
import torch
import numpy as np
print("NumPy:", np.__version__)
import gridding, gridding_distance, chamfer, cubic_feature_sampling
from pointnet2_ops import pointnet2_utils
from models import AdaPoinTr
print("✅ Her sey calisiyor!")

/content/PoinTr
NumPy: 1.26.4


[transformers] Disabling PyTorch because PyTorch >= 2.4 is required but found 2.2.2+cu118
[transformers] PyTorch was not found. Models won't be available and only tokenizers, configuration and file/data utilities can be used.


✅ Her sey calisiyor!


/usr/local/lib/python3.12/dist-packages/timm/models/layers/__init__.py:49: FutureWarning: Importing from timm.models.layers is deprecated, please import via timm.layers
  warnings.warn(f"Importing from {__name__} is deprecated, please import via timm.layers", FutureWarning)


In [ ]:
%cd /content
!mv PoinTr original_PoinTr
!git clone https://github.com/eylulpelinkilic/Pelin_Efe_PoinTr.git PoinTr
%cd PoinTr
!git rev-parse HEAD

/content
Cloning into 'PoinTr'...
remote: Enumerating objects: 595, done.
remote: Counting objects: 100% (368/368), done.
remote: Compressing objects: 100% (135/135), done.
remote: Total 595 (delta 297), reused 233 (delta 233), pack-reused 227 (from 2)
Receiving objects: 100% (595/595), 26.29 MiB | 29.49 MiB/s, done.
Resolving deltas: 100% (337/337), done.
/content/PoinTr
3f98676aeaabd9c1150dd04cd0a51f504949ab9c


In [ ]:
import os
os.environ["CUDA_HOME"] = "/usr/local/cuda-11.8"
os.environ["PATH"] = "/usr/local/cuda-11.8/bin:" + os.environ["PATH"]
%cd /content/PoinTr
import torch
import gridding, gridding_distance, chamfer, cubic_feature_sampling
from pointnet2_ops import pointnet2_utils
from models import AdaPoinTr
print("✅ Fork ile de calisiyor!")

/content/PoinTr
✅ Fork ile de calisiyor!


In [ ]:
try:
    import pointnet2_ops; print("pointnet2_ops OK")
except Exception as e:
    print("YOK, derlenmeli:", e)
    # setup zaten kurduysa /content/PoinTr/... içindedir; yoksa:
    # pip install "git+https://github.com/erikwijmans/Pointnet2_PyTorch.git#subdirectory=pointnet2_ops_lib"


pointnet2_ops OK


In [ ]:
import os, glob, subprocess
# 1) diskte var mı, ara
hits = glob.glob("/content/**/PoinTr_ShapeNet55.pth", recursive=True) + \
       glob.glob("/content/**/*ShapeNet55*.pth", recursive=True)
print("bulunanlar:", hits)

CKPT = "/content/PoinTr/ckpts/PoinTr_ShapeNet55.pth"
if hits and os.path.getsize(hits[0]) > 50e6:
    os.makedirs(os.path.dirname(CKPT), exist_ok=True)
    if hits[0] != CKPT: subprocess.run(f'cp "{hits[0]}" "{CKPT}"', shell=True)
    print("kopyalandı ->", CKPT)
else:
    os.makedirs(os.path.dirname(CKPT), exist_ok=True)
    subprocess.run("pip install -q gdown", shell=True)
    subprocess.run(f"gdown 1WzERLlbSwzGOBybzkjBrApwyVMTG00CJ -O {CKPT}", shell=True, check=True)

print("boyut MB:", round(os.path.getsize(CKPT)/1e6, 1), " (>400 olmalı)")


bulunanlar: []
boyut MB: 415.9  (>400 olmalı)


In [ ]:
import os, sys, glob, numpy as np, torch, torch.nn as nn, open3d as o3d
from scipy.spatial import cKDTree
sys.path.insert(0, "/content/PoinTr"); os.chdir("/content/PoinTr")
from easydict import EasyDict
from models.PoinTr import PoinTr           # import chamfer ext'i çeker; env setup derlemiş olmalı

# ---------------- CONFIG ----------------
CKPT     = "/content/PoinTr/ckpts/PoinTr_ShapeNet55.pth"
GT_DIR   = "/content/colored_gt"           # <-- renkli 8192-nokta GT PLY'lerin burada
NCLOUDS  = 50                              # kaç bulut kullanılacak
NTRAIN   = 40                              # ilk NTRAIN eğitim, kalanı val
CROP     = 0.5                             # occlusion oranı
SEED     = 42
DEV      = "cuda"
# ----------------------------------------

cfg = EasyDict(trans_dim=384, knn_layer=1, num_pred=6144, num_query=96)
model = PoinTr(cfg)
sd = torch.load(CKPT, map_location="cpu")
key = "base_model" if "base_model" in sd else ("model" if "model" in sd else None)
base = sd[key] if key else sd
base = {k.replace("module.", ""): v for k, v in base.items()}
miss, unexp = model.load_state_dict(base, strict=False)
print(f"checkpoint yüklendi | missing={len(miss)} unexpected={len(unexp)}  (ikisi de ~0 olmalı)")
for p in model.parameters(): p.requires_grad_(False)
model.eval().to(DEV)

# forward hook -> latent (rebuild_feature, B*M x 384)
_lat = {}
model.reduce_map.register_forward_hook(lambda m, i, o: _lat.__setitem__("rf", o.detach()))

# ---- helpers ----
def load_ply(p):
    pc = o3d.io.read_point_cloud(p)
    xyz = np.asarray(pc.points); rgb = np.asarray(pc.colors)
    if len(rgb) != len(xyz): rgb = np.zeros_like(xyz)
    return np.concatenate([xyz, rgb], 1).astype(np.float32)

def pc_norm(gt):                            # unit-sphere (GT frame; PoinTr'ın eğitim konvansiyonu)
    x = gt[:, :3] - gt[:, :3].mean(0)
    x = x / (np.linalg.norm(x, axis=1).max() + 1e-9)
    return np.concatenate([x, gt[:, 3:6]], 1).astype(np.float32)

def separate_colored(gt, crop=CROP, seed=0):   # -> partial(6), missing_mask over gt
    xyz = gt[:, :3]; N = len(gt); nc = int(round(N * crop))
    c = xyz.mean(0); xyzn = (xyz - c) / (np.linalg.norm(xyz - c, axis=1).max() + 1e-9)
    rng = np.random.default_rng(seed); v = rng.standard_normal(3); v /= np.linalg.norm(v)
    order = np.argsort(np.linalg.norm(xyzn - v[None], axis=1))
    mask = np.zeros(N, bool); mask[order[:nc]] = True   # True = missing (occluded)
    return gt[order[nc:]], mask

def srgb_to_lab(rgb):
    rgb = np.clip(rgb, 0, 1); lin = np.where(rgb > 0.04045, ((rgb + 0.055) / 1.055) ** 2.4, rgb / 12.92)
    M = np.array([[0.4124, 0.3576, 0.1805], [0.2126, 0.7152, 0.0722], [0.0193, 0.1192, 0.9505]])
    xyz = (lin @ M.T) / np.array([0.95047, 1.0, 1.08883]); d = 6 / 29
    f = np.where(xyz > d ** 3, np.cbrt(xyz), xyz / (3 * d ** 2) + 4 / 29)
    return np.stack([116 * f[:, 1] - 16, 500 * (f[:, 0] - f[:, 1]), 200 * (f[:, 1] - f[:, 2])], 1)
def deltaE(a, b): return np.linalg.norm(srgb_to_lab(a) - srgb_to_lab(b), axis=1)
print("hazır.")


2026-07-06 02:36:51,227 - MODEL - INFO -  Transformer with knn_layer 1


checkpoint yüklendi | missing=0 unexpected=0  (ikisi de ~0 olmalı)
hazır.


In [ ]:
import glob
print(len(glob.glob("/content/colored_gt/*.ply") + glob.glob("/content/colored_gt/*/*.ply")), "PLY bulundu")


0 PLY bulundu


In [ ]:
def load_ply(p, n=8192):
    pc = o3d.io.read_point_cloud(p)
    if len(pc.points) > n:
        pc = pc.farthest_point_down_sample(n)       # uniform FPS -> 8192
    xyz = np.asarray(pc.points); rgb = np.asarray(pc.colors)
    if len(rgb) != len(xyz): rgb = np.zeros_like(xyz)
    return np.concatenate([xyz, rgb], 1).astype(np.float32)
print("load_ply guncellendi (FPS'li)")


load_ply guncellendi (FPS'li)


In [ ]:
from google.colab import files
import zipfile, os, glob
os.makedirs("/content/colored_gt", exist_ok=True)
up = files.upload()                          # Masaüstünden colored_gt.zip sec
with zipfile.ZipFile(list(up.keys())[0]) as f: f.extractall("/content/colored_gt")
print(len(glob.glob("/content/colored_gt/*.ply")), "PLY hazir")   # 45 gormeli


Saving colored_gt.zip to colored_gt.zip
45 PLY hazir


In [ ]:
import glob, random, numpy as np, torch, open3d as o3d
from scipy.spatial import cKDTree
GT_DIR = "/content/colored_gt"

def load_ply(p, n=8192):                        # 8192'den buyukse FPS, degilse oldugu gibi
    pc = o3d.io.read_point_cloud(p)
    if len(pc.points) > n: pc = pc.farthest_point_down_sample(n)
    xyz = np.asarray(pc.points); rgb = np.asarray(pc.colors)
    if len(rgb) != len(xyz): rgb = np.zeros_like(xyz)
    return np.concatenate([xyz, rgb], 1).astype(np.float32)

plys = sorted(glob.glob(f"{GT_DIR}/*.ply") + glob.glob(f"{GT_DIR}/*/*.ply"))[:NCLOUDS]
random.Random(0).shuffle(plys)                  # kategorileri karistir (val tek kategori olmasin)
assert plys, f"{GT_DIR} altinda PLY yok"
print(f"{len(plys)} bulut")

NP_, NQ = model.num_pred, model.num_query; S = NP_ // NQ
DATA = []
for j, p in enumerate(plys):
    gt = pc_norm(load_ply(p))                    # unit-sphere (GT frame)
    partial, mmask = separate_colored(gt, seed=SEED + j)
    with torch.no_grad():
        pin = torch.from_numpy(partial[:, :3]).float().unsqueeze(0).to(DEV)
        _lat.clear(); ret = model(pin)
    folded = ret[1][0, :NP_].cpu().numpy()        # (6144,3) tamamlanan yeni noktalar
    rf = _lat["rf"].reshape(NQ, -1).cpu().numpy() # (96,384) token latent
    latent_pp = np.repeat(rf, S, axis=0).astype(np.float32)   # (6144,384) hizali
    _, gi = cKDTree(gt[:, :3]).query(folded, k=1)
    target = gt[gi, 3:6]; is_miss = mmask[gi]     # hedef renk + eksik-bolge maskesi
    ptree = cKDTree(partial[:, :3]); _, pi = ptree.query(folded, k=1)
    nn_rgb = partial[pi, 3:6]                      # NN-kopya baseline
    kk = min(8, len(partial)); _, ci = ptree.query(folded, k=kk)
    ctx = partial[ci, 3:6].mean(1) if kk > 1 else partial[ci, 3:6]   # gozlenen renk baglami
    DATA.append(dict(folded=folded, latent=latent_pp, ctx=ctx.astype(np.float32),
                     target=target.astype(np.float32), nn=nn_rgb.astype(np.float32),
                     miss=is_miss, gt=gt, partial=partial))
    if j % 10 == 0: print(f"  {j+1}/{len(plys)}  missing-orani≈{is_miss.mean():.2f}")
print("cache hazir.")


45 bulut
  1/45  missing-orani≈0.61
  11/45  missing-orani≈0.62
  21/45  missing-orani≈0.70
  31/45  missing-orani≈0.69
  41/45  missing-orani≈0.78
cache hazir.


In [ ]:
import numpy as np, torch, torch.nn as nn, open3d as o3d
NTRAIN = 35                                       # 35 egitim / 10 val
tr, va = DATA[:NTRAIN], DATA[NTRAIN:]
cat = lambda ds, k: np.concatenate([d[k] for d in ds], 0)
Xc = np.concatenate([cat(tr,"folded"), cat(tr,"ctx")], 1)   # [xyz, ctx]  -> C
Xd = np.concatenate([cat(tr,"latent"), cat(tr,"ctx")], 1)   # [latent, ctx] -> D
Y  = cat(tr,"target")

class Head(nn.Module):
    def __init__(s, d): super().__init__(); s.n = nn.Sequential(
        nn.Linear(d,256), nn.ReLU(), nn.Linear(256,256), nn.ReLU(), nn.Linear(256,3), nn.Sigmoid())
    def forward(s, x): return s.n(x)

def train(head, X, Y, ep=40, bs=16384):
    head.to(DEV); opt = torch.optim.Adam(head.parameters(), 1e-3)
    Xt, Yt = torch.from_numpy(X).float().to(DEV), torch.from_numpy(Y).float().to(DEV)
    for e in range(ep):
        perm = torch.randperm(len(Xt), device=DEV)
        for i in range(0, len(Xt), bs):
            idx = perm[i:i+bs]; opt.zero_grad()
            l = (head(Xt[idx]) - Yt[idx]).abs().mean(); l.backward(); opt.step()
        if e % 10 == 0: print(f"  ep{e} L1 {l.item():.4f}")
    return head

print("Head-C (koordinat):"); headC = train(Head(Xc.shape[1]), Xc, Y)
print("Head-D (latent):");    headD = train(Head(Xd.shape[1]), Xd, Y)

Xc_v = np.concatenate([cat(va,"folded"), cat(va,"ctx")], 1)
Xd_v = np.concatenate([cat(va,"latent"), cat(va,"ctx")], 1)
Y_v, NN_v, M_v = cat(va,"target"), cat(va,"nn"), cat(va,"miss")
with torch.no_grad():
    Pc = headC(torch.from_numpy(Xc_v).float().to(DEV)).cpu().numpy()
    Pd = headD(torch.from_numpy(Xd_v).float().to(DEV)).cpu().numpy()
m = M_v
print("\n=== EKSIK BOLGE ortalama renk dE (val, dusuk=iyi) ===")
print(f"  NN-kopya : {deltaE(NN_v[m], Y_v[m]).mean():.3f}")
print(f"  Head-C   : {deltaE(Pc[m],  Y_v[m]).mean():.3f}   (koordinat)")
print(f"  Head-D   : {deltaE(Pd[m],  Y_v[m]).mean():.3f}   (latent)")
print(f"  [collapse kontrol] pred std C={Pc.std(0).mean():.3f} D={Pd.std(0).mean():.3f} GT={Y_v.std(0).mean():.3f}")

d0 = va[0]; f0 = d0["folded"]
with torch.no_grad():
    pd0 = headD(torch.from_numpy(np.concatenate([d0["latent"], d0["ctx"]],1)).float().to(DEV)).cpu().numpy()
def save(xyz, rgb, path):
    pc = o3d.geometry.PointCloud(); pc.points = o3d.utility.Vector3dVector(xyz.astype(np.float64))
    pc.colors = o3d.utility.Vector3dVector(np.clip(rgb,0,1).astype(np.float64)); o3d.io.write_point_cloud(path, pc)
save(d0["gt"][:,:3], d0["gt"][:,3:6], "/content/eval_gt.ply")
save(f0, d0["nn"], "/content/eval_nncopy.ply")
save(f0, pd0,      "/content/eval_headD.ply")
print("\nkaydedildi: eval_gt.ply / eval_nncopy.ply / eval_headD.ply")


Head-C (koordinat):
  ep0 L1 0.2599
  ep10 L1 0.1667
  ep20 L1 0.1650
  ep30 L1 0.1676
Head-D (latent):
  ep0 L1 0.2590
  ep10 L1 0.1465
  ep20 L1 0.1348
  ep30 L1 0.1371

=== EKSIK BOLGE ortalama renk dE (val, dusuk=iyi) ===
  NN-kopya : 24.320
  Head-C   : 23.591   (koordinat)
  Head-D   : 38.086   (latent)
  [collapse kontrol] pred std C=0.342 D=0.317 GT=0.376

kaydedildi: eval_gt.ply / eval_nncopy.ply / eval_headD.ply


In [ ]:
class Head(nn.Module):
    def __init__(s, d, p=0.3): super().__init__(); s.n = nn.Sequential(
        nn.Linear(d,128), nn.ReLU(), nn.Dropout(p), nn.Linear(128,3), nn.Sigmoid())
    def forward(s, x): return s.n(x)
def train(head, X, Y, ep=40, bs=16384, wd=1e-2):
    head.to(DEV); opt = torch.optim.Adam(head.parameters(), 1e-3, weight_decay=wd)
    Xt, Yt = torch.from_numpy(X).float().to(DEV), torch.from_numpy(Y).float().to(DEV)
    for e in range(ep):
        perm = torch.randperm(len(Xt), device=DEV)
        for i in range(0, len(Xt), bs):
            idx = perm[i:i+bs]; opt.zero_grad()
            l = (head(Xt[idx]) - Yt[idx]).abs().mean(); l.backward(); opt.step()
    return head.eval()
headC = train(Head(Xc.shape[1]), Xc, Y); headD = train(Head(Xd.shape[1]), Xd, Y)
with torch.no_grad():
    Pc = headC(torch.from_numpy(Xc_v).float().to(DEV)).cpu().numpy()
    Pd = headD(torch.from_numpy(Xd_v).float().to(DEV)).cpu().numpy()
    # train ΔE (overfitting kontrolu)
    Pd_tr = headD(torch.from_numpy(Xd).float().to(DEV)).cpu().numpy()
m = M_v
print(f"NN-kopya : {deltaE(NN_v[m],Y_v[m]).mean():.2f}")
print(f"Head-C   : {deltaE(Pc[m], Y_v[m]).mean():.2f}")
print(f"Head-D   : {deltaE(Pd[m], Y_v[m]).mean():.2f}   (val)")
print(f"Head-D   : {deltaE(Pd_tr, Y).mean():.2f}   (TRAIN — val'den cok dusukse overfit dogrulandi)")


NN-kopya : 24.32
Head-C   : 29.79
Head-D   : 36.85   (val)
Head-D   : 23.57   (TRAIN — val'den cok dusukse overfit dogrulandi)


In [ ]:
import numpy as np, open3d as o3d, plotly.graph_objects as go
from plotly.subplots import make_subplots
g = np.asarray(o3d.io.read_point_cloud("/content/eval_gt.ply").points)
c = np.asarray(o3d.io.read_point_cloud("/content/eval_nncopy.ply").points)
fig = make_subplots(rows=1, cols=2, specs=[[{"type":"scene"}]*2], subplot_titles=("GT","completion"))
fig.add_trace(go.Scatter3d(x=g[:,0],y=g[:,1],z=g[:,2],mode="markers",marker=dict(size=1.5)),1,1)
fig.add_trace(go.Scatter3d(x=c[:,0],y=c[:,1],z=c[:,2],mode="markers",marker=dict(size=1.5)),1,2)
fig.layout.scene.aspectmode="data"; fig.layout.scene2.aspectmode="data"; fig.show()


In [ ]:
import torch, torch.nn as nn
from easydict import EasyDict
from models.dgcnn_group import DGCNN_Grouper
from models.Transformer import PCTransformer
from models.PoinTr import PoinTr, Fold, fps

# --- grouper: orijinalin BIREBIR kopyasi (input_trans 6D), sarmalama yok ---
def grouper_init(self):
    nn.Module.__init__(self)
    self.input_trans = nn.Conv1d(6, 8, 1)
    self.layer1 = nn.Sequential(nn.Conv2d(16,32,1,bias=False),  nn.GroupNorm(4,32),  nn.LeakyReLU(0.2))
    self.layer2 = nn.Sequential(nn.Conv2d(64,64,1,bias=False),  nn.GroupNorm(4,64),  nn.LeakyReLU(0.2))
    self.layer3 = nn.Sequential(nn.Conv2d(128,64,1,bias=False), nn.GroupNorm(4,64),  nn.LeakyReLU(0.2))
    self.layer4 = nn.Sequential(nn.Conv2d(128,128,1,bias=False),nn.GroupNorm(4,128), nn.LeakyReLU(0.2))
def grouper_forward(self, x):
    coor = x[:, :3].contiguous(); f = self.input_trans(x)
    f = self.get_graph_feature(coor,f,coor,f); f=self.layer1(f); f=f.max(-1)[0]
    cq,fq = self.fps_downsample(coor,f,512); f=self.get_graph_feature(cq,fq,coor,f); f=self.layer2(f); f=f.max(-1)[0]; coor=cq
    f = self.get_graph_feature(coor,f,coor,f); f=self.layer3(f); f=f.max(-1)[0]
    cq,fq = self.fps_downsample(coor,f,128); f=self.get_graph_feature(cq,fq,coor,f); f=self.layer4(f); f=f.max(-1)[0]; coor=cq
    return coor, f
DGCNN_Grouper.__init__ = grouper_init
DGCNN_Grouper.forward  = grouper_forward

# --- PoinTr: orijinalin BIREBIR kopyasi + color_head, sarmalama yok ---
def poinTr_init(self, config, **kw):
    nn.Module.__init__(self)
    self.trans_dim=config.trans_dim; self.knn_layer=config.knn_layer
    self.num_pred=config.num_pred;  self.num_query=config.num_query
    self.fold_step=int(pow(self.num_pred//self.num_query,0.5)+0.5)
    self.base_model=PCTransformer(in_chans=3, embed_dim=self.trans_dim, depth=[6,8], drop_rate=0., num_query=self.num_query, knn_layer=self.knn_layer)
    self.foldingnet=Fold(self.trans_dim, step=self.fold_step, hidden_dim=256)
    self.color_head=Fold(self.trans_dim, step=self.fold_step, hidden_dim=256)
    self.increase_dim=nn.Sequential(nn.Conv1d(self.trans_dim,1024,1), nn.BatchNorm1d(1024), nn.LeakyReLU(0.2), nn.Conv1d(1024,1024,1))
    self.reduce_map=nn.Linear(self.trans_dim+1027, self.trans_dim)
def poinTr_forward(self, xyz):
    q, coarse = self.base_model(xyz); B,M,C = q.shape
    gf = self.increase_dim(q.transpose(1,2)).transpose(1,2); gf = torch.max(gf,1)[0]
    rf = self.reduce_map(torch.cat([gf.unsqueeze(-2).expand(-1,M,-1), q, coarse],-1).reshape(B*M,-1))
    reb_xyz = (self.foldingnet(rf).reshape(B,M,3,-1) + coarse.unsqueeze(-1)).transpose(2,3).reshape(B,-1,3)
    reb_rgb = torch.sigmoid(self.color_head(rf)).reshape(B,M,3,-1).transpose(2,3).reshape(B,-1,3)
    reb = torch.cat([reb_xyz, reb_rgb], -1)
    inp = fps(xyz[:,:,:3].contiguous(), self.num_query)
    return torch.cat([coarse, inp],1), torch.cat([reb, xyz],1)
PoinTr.__init__ = poinTr_init
PoinTr.forward  = poinTr_forward

cfg = EasyDict(trans_dim=384, knn_layer=1, num_pred=6144, num_query=96)
model_c = PoinTr(cfg)
for k in ["base_model.grouper.input_trans.weight","base_model.grouper.input_trans.bias"]:
    base.pop(k, None)
mi, ui = model_c.load_state_dict(base, strict=False)
print(f"OK | missing={len(mi)} unexpected={len(ui)}")
model_c.to(DEV)


2026-07-06 03:38:36,826 - MODEL - INFO -  Transformer with knn_layer 1


OK | missing=30 unexpected=0


PoinTr(
  (base_model): PCTransformer(
    (grouper): DGCNN_Grouper(
      (input_trans): Conv1d(6, 8, kernel_size=(1,), stride=(1,))
      (layer1): Sequential(
        (0): Conv2d(16, 32, kernel_size=(1, 1), stride=(1, 1), bias=False)
        (1): GroupNorm(4, 32, eps=1e-05, affine=True)
        (2): LeakyReLU(negative_slope=0.2)
      )
      (layer2): Sequential(
        (0): Conv2d(64, 64, kernel_size=(1, 1), stride=(1, 1), bias=False)
        (1): GroupNorm(4, 64, eps=1e-05, affine=True)
        (2): LeakyReLU(negative_slope=0.2)
      )
      (layer3): Sequential(
        (0): Conv2d(128, 64, kernel_size=(1, 1), stride=(1, 1), bias=False)
        (1): GroupNorm(4, 64, eps=1e-05, affine=True)
        (2): LeakyReLU(negative_slope=0.2)
      )
      (layer4): Sequential(
        (0): Conv2d(128, 128, kernel_size=(1, 1), stride=(1, 1), bias=False)
        (1): GroupNorm(4, 128, eps=1e-05, affine=True)
        (2): LeakyReLU(negative_slope=0.2)
      )
    )
    (pos_embed): Sequent

In [ ]:
import numpy as np, plotly.graph_objects as go
from plotly.subplots import make_subplots

def loss_fn(fine6, gt6, ns=4096, cw=10.0):
    f, g = fine6[0], gt6[0]
    i = torch.randperm(f.shape[0], device=DEV)[:ns]; fs = f[i]
    D = torch.cdist(fs[:,:3], g[:,:3]); d1, nn_ = D.min(1); d2 = D.min(0)[0]
    cham = d1.mean() + d2.mean()
    col = (fs[:,3:6] - g[nn_,3:6]).abs().mean()
    return cham + cw*col, cham.item(), col.item()

subset = DATA[:5]                                          # 5 bulutu ezberle
model_c.train(); opt = torch.optim.Adam(model_c.parameters(), 1e-4)
for step in range(400):
    d = subset[step % len(subset)]
    p6 = torch.from_numpy(d["partial"]).float().unsqueeze(0).to(DEV)
    g6 = torch.from_numpy(d["gt"]).float().unsqueeze(0).to(DEV)
    _, fine6 = model_c(p6); loss, ch, co = loss_fn(fine6, g6)
    opt.zero_grad(); loss.backward(); opt.step()
    if step % 50 == 0: print(f"step {step}  chamfer={ch:.4f}  color={co:.4f}")

# --- gorsel: overfit ettigimiz bir bulut ---
model_c.eval()
d = subset[0]
with torch.no_grad():
    _, fine6 = model_c(torch.from_numpy(d["partial"]).float().unsqueeze(0).to(DEV))
fine = fine6[0].cpu().numpy(); gt = d["gt"]
def tr(a):
    col = ["rgb(%d,%d,%d)"%(int(r*255),int(g*255),int(b*255)) for r,g,b in np.clip(a[:,3:6],0,1)]
    return go.Scatter3d(x=a[:,0],y=a[:,1],z=a[:,2],mode="markers",marker=dict(size=1.6,color=col))
fig = make_subplots(rows=1,cols=2,specs=[[{"type":"scene"}]*2],subplot_titles=("GT","6D PoinTr completion (colored)"))
fig.add_trace(tr(gt),1,1); fig.add_trace(tr(fine),1,2)
fig.layout.scene.aspectmode="data"; fig.layout.scene2.aspectmode="data"; fig.update_layout(height=500,showlegend=False)
fig.show()


step 0  chamfer=0.2385  color=0.2148
step 50  chamfer=0.0363  color=0.1655
step 100  chamfer=0.0260  color=0.1378
step 150  chamfer=0.0246  color=0.1180
step 200  chamfer=0.0222  color=0.1262
step 250  chamfer=0.0218  color=0.1186
step 300  chamfer=0.0335  color=0.1159
step 350  chamfer=0.0218  color=0.1052


In [ ]:
import numpy as np, torch, plotly.graph_objects as go
from plotly.subplots import make_subplots

model_c.eval()
subset = DATA[:3]                                  # az bulut -> daha temiz overfit
model_c.train(); opt = torch.optim.Adam(model_c.parameters(), 2e-4)
for step in range(1000):
    d = subset[step % len(subset)]
    p6 = torch.from_numpy(d["partial"]).float().unsqueeze(0).to(DEV)
    g6 = torch.from_numpy(d["gt"]).float().unsqueeze(0).to(DEV)
    _, fine6 = model_c(p6)
    loss, ch, co = loss_fn(fine6, g6, cw=1.0)      # cw 10 -> 1: once geometri
    opt.zero_grad(); loss.backward(); opt.step()
    if step % 100 == 0: print(f"step {step}  chamfer={ch:.4f}  color={co:.4f}")

def tr(a):
    col = ["rgb(%d,%d,%d)" % (int(r*255),int(g*255),int(b*255)) for r,g,b in np.clip(a[:,3:6],0,1)]
    return go.Scatter3d(x=a[:,0],y=a[:,1],z=a[:,2],mode="markers",marker=dict(size=1.6,color=col))

fig = make_subplots(rows=1, cols=3, specs=[[{"type":"scene"}]*3],
                    subplot_titles=("occluded (input)", "6D PoinTr completion", "GT"))
fig.add_trace(tr(partial), 1, 1)
fig.add_trace(tr(fine),    1, 2)
fig.add_trace(tr(gt),      1, 3)
for s in ("scene","scene2","scene3"): fig.layout[s].aspectmode = "data"
fig.update_layout(height=500, showlegend=False, margin=dict(l=0,r=0,t=30,b=0))
fig.show()


step 0  chamfer=0.0240  color=0.1086
step 100  chamfer=0.0257  color=0.0969
step 200  chamfer=0.0242  color=0.0500
step 300  chamfer=0.0164  color=0.1091
step 400  chamfer=0.0250  color=0.0857
step 500  chamfer=0.0200  color=0.0402
step 600  chamfer=0.0164  color=0.1160
step 700  chamfer=0.0247  color=0.0782
step 800  chamfer=0.0218  color=0.0609
step 900  chamfer=0.0138  color=0.1208


In [ ]:
import numpy as np, torch, torch.nn as nn, plotly.graph_objects as go
from plotly.subplots import make_subplots

demo = DATA[:3]                                          # 3 bulutu sıkı ezberle
Xtr = np.concatenate([np.concatenate([d["latent"], d["ctx"]],1) for d in demo],0)
Ytr = np.concatenate([d["target"] for d in demo],0)

class ColorHead(nn.Module):
    def __init__(s,d): super().__init__(); s.n=nn.Sequential(
        nn.Linear(d,256),nn.ReLU(),nn.Linear(256,256),nn.ReLU(),nn.Linear(256,3),nn.Sigmoid())
    def forward(s,x): return s.n(x)

head = ColorHead(Xtr.shape[1]).to(DEV)
opt = torch.optim.Adam(head.parameters(), 1e-3)
Xt, Yt = torch.from_numpy(Xtr).float().to(DEV), torch.from_numpy(Ytr).float().to(DEV)
for e in range(800):
    opt.zero_grad(); l=(head(Xt)-Yt).abs().mean(); l.backward(); opt.step()
    if e%100==0: print("ep",e,"L1",round(l.item(),4))           # dusmeli (0.2 -> dusuk)

# --- 3 panel: occluded / completion (frozen geom + ogrenilen renk) / GT ---
d = demo[0]
with torch.no_grad():
    pred = head(torch.from_numpy(np.concatenate([d["latent"],d["ctx"]],1)).float().to(DEV)).cpu().numpy()
comp = np.concatenate([d["folded"], pred], 1)
def tr(a):
    c=["rgb(%d,%d,%d)"%(int(r*255),int(g*255),int(b*255)) for r,g,b in np.clip(a[:,3:6],0,1)]
    return go.Scatter3d(x=a[:,0],y=a[:,1],z=a[:,2],mode="markers",marker=dict(size=1.6,color=c))
fig = make_subplots(rows=1,cols=3,specs=[[{"type":"scene"}]*3],
                    subplot_titles=("occluded (input)","completion (color head)","GT"))
fig.add_trace(tr(d["partial"]),1,1); fig.add_trace(tr(comp),1,2); fig.add_trace(tr(d["gt"]),1,3)
for s in ("scene","scene2","scene3"): fig.layout[s].aspectmode="data"
fig.update_layout(height=500,showlegend=False,margin=dict(l=0,r=0,t=30,b=0)); fig.show()

# istersen PLY kaydet:
# import open3d as o3d
# for name,a in [("occluded",d["partial"]),("completion",comp),("gt",d["gt"])]:
#     pc=o3d.geometry.PointCloud(); pc.points=o3d.utility.Vector3dVector(a[:,:3].astype(np.float64))
#     pc.colors=o3d.utility.Vector3dVector(np.clip(a[:,3:6],0,1).astype(np.float64))
#     o3d.io.write_point_cloud(f"/content/final_{name}.ply",pc)


NameError: name 'DATA' is not defined